In [18]:
#@title MyDriveのマウント

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
#@title 年と月を'選択してください { run: "auto" }
selected_year = '2026' # @param ['2026', '2027', '2028'] {type:"string"}

selected_month = "02" # @param ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'] {type:"string"}

year_month = f"{selected_year}-{selected_month}"
print( f"選択された年と月：{year_month}")

選択された年と月：2026-02


In [20]:
#@title 'MyDrive/receipts/処理結果/年-月/csv'ディレクトリ内のcsvファイルへのパスを取得

import os

# パスの指定
path = f'/content/drive/MyDrive/receipts/処理結果/{year_month}/csv'

# ファイル名のリストを取得
file_names = os.listdir(path)

# 取得したファイル名を表示
for file_name in file_names:
    print(file_name)

print()

# フルパスに変換してリストに格納
csv_paths = [os.path.join(path, f) for f in os.listdir(path)]  # 内包表記を利用
print(csv_paths)

20260317_1 - 領収書解析結果とCSV作成.csv
20260317_2 - 領収書解析結果の構造化リスト.csv

['/content/drive/MyDrive/receipts/処理結果/2026-02/csv/20260317_1 - 領収書解析結果とCSV作成.csv', '/content/drive/MyDrive/receipts/処理結果/2026-02/csv/20260317_2 - 領収書解析結果の構造化リスト.csv']


In [27]:
#@title 試しにpandasで読み込んでみる

import pandas as pd

# 先ほど取得したリストの1番目（インデックス0）を読み込む
df = pd.read_csv(csv_paths[0])

# 内容の確認
print(f"読み込んだファイル: {csv_paths[0]}")
display(df.head())

読み込んだファイル: /content/drive/MyDrive/receipts/処理結果/2026-02/csv/20260317_1 - 領収書解析結果とCSV作成.csv


,日付,科目,摘要,支出金額,処理対象ファイル名
0,2026/02/01,会議費,酒屋いって,22400,Image_20260317_0002.pdf
1,2026/02/05,図書研修費,協会けんぽ一般(胸直背直) 他,3686,Image_20260317_0002.pdf
2,2026/02/17,旅行交通費,つばめタクシー(株),1200,Image_20260317_0002.pdf
3,2026/02/20,旅行交通費,ホテル法華クラブ新潟長岡,10300,Image_20260317_0002.pdf
4,2026/02/20,旅行交通費,中越交通株式会社,1400,Image_20260317_0002.pdf


In [22]:
#@title 複数のcsvを読み込んで結合する

# 各ファイルを順番に読み込んでリストに格納
df_list = []
for path in csv_paths:
    temp_df = pd.read_csv(path)
    df_list.append(temp_df)

# すべてのDataFrameを縦方向に結合
df_concat = pd.concat(df_list, ignore_index=True)

# 結果の確認
print(f"結合したファイル数: {len(df_list)}")
print()
print(f"結合した領収書数: {len(df_concat)}")
print(len(df_concat))
print()

print('最初の3つ')
display(df_concat.head(3))
print()
print('最後の3つ')
display(df_concat.tail(3))

結合したファイル数: 2

結合した領収書数: 21
21

最初の3つ


,日付,科目,摘要,支出金額,処理対象ファイル名
0,2026/02/01,会議費,酒屋いって,22400,Image_20260317_0002.pdf
1,2026/02/05,図書研修費,協会けんぽ一般(胸直背直) 他,3686,Image_20260317_0002.pdf
2,2026/02/17,旅行交通費,つばめタクシー(株),1200,Image_20260317_0002.pdf



最後の3つ


,日付,科目,摘要,支出金額,処理対象ファイル名
18,2026/02/26,事務消耗品費,パンチ状差し黒 SF-100-BK +4,1154,Image_20260317_0001.pdf
19,2026/02/26,図書研修費,定期健康診断(胸部間接) +1,8800,Image_20260317_0001.pdf
20,2026/02/28,衛生管理費,株式会社ウェットランドリー +2,2101,Image_20260317_0001.pdf


In [26]:
#@title 結合したdf_concatの要素を適切な型に変換する
# 1. 日付列を datetime型に変換
df_concat['日付'] = pd.to_datetime(df_concat['日付'])

# 2. 金額・税額列から数値以外（カンマ等）を除去して整数型(int)に変換
# ※数値として読み取れない値（欠損値など）がある場合は .fillna(0) などで補完が必要
df_concat['支出金額'] = df_concat['支出金額'].replace({',': '', '¥': ''}, regex=True).astype(int)

# 変換後の型を確認
print(df_concat[['日付', '支出金額']].dtypes)
print()

display(df_concat.head())

ValueError: invalid literal for int() with base 10: '1342 +1'

In [25]:
#@title 科目とコードの関係を辞書で定義

code_book = {
    "運賃": 511,
    "支払手数料": 512,
    "諸会費": 520,
    "接待交際費": 521,
    "旅行交通費": 522,
    "通信費": 523,
    "事務消耗品費": 524,
    "消耗品費": 525,
    "租税公課": 526,
    "修繕費": 529,
    "保険料": 531,
    "燃料費": 533,
    "寄付金": 535,
    "管理諸費": 536,
    "図書研修費": 537,
    "雑費": 538,
    "医薬品費": 401,
    "衛生管理費": 444
}

In [ ]:
#@title '科目'列のインデックスを取得して、その右隣（+1）に'コード列を挿入
col_idx = df_concat.columns.get_loc('科目') + 1
df_concat.insert(col_idx, 'コード', df_concat['科目'].map(code_book))
display(df_concat.head())

In [ ]:
#@title year_month以前の日付は、year_monthに変更する。元の日付は摘要に(m月d日分)として追加する。

# year_monthをdatetime型に変換（その月の1日として設定）
target_date = pd.to_datetime(year_month + '-01')

# 条件：日付がターゲット月より前であるか判定
mask = df_concat['日付'] < target_date

# 1. 摘要の更新（「月日」の形式で追記）
df_concat.loc[mask, '摘要'] = (
    df_concat.loc[mask, '摘要'] +
    '(' + df_concat.loc[mask, '日付'].dt.month.astype(str) + '月' +
    df_concat.loc[mask, '日付'].dt.day.astype(str) + '日分)'
)

# 2. 該当する日付をyear_monthに上書き
df_concat.loc[mask, '日付'] = target_date

display(df_concat)

In [24]:
#@title 編集したdf_concatをcsvファイルとして保存する

# 保存先のディレクトリパス
output_dir = f'/content/drive/MyDrive/receipts/処理結果/{year_month}/結合済みcsv'
file_name = f'{year_month}.csv'

# 保存先のフルパスを作成
save_path = os.path.join(output_dir, file_name)

# CSVとして保存
# index=False にすることで、DataFrameのインデックス（左端の数字列）を除外して保存できます
df_concat.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"{file_name}の保存が完了しました: {save_path}")

2026-02.csvの保存が完了しました: /content/drive/MyDrive/receipts/処理結果/2026-02/結合済みcsv/2026-02.csv
